In [9]:
from datapipeline.data.load_data import load_raw_data
import yaml
import pandas as pd
import mlflow
from pathlib import Path
from sklearn.metrics import (roc_auc_score, 
                            balanced_accuracy_score,
                            precision_score,
                            recall_score,
                            f1_score,
                            confusion_matrix,
                            precision_recall_curve,
                            average_precision_score,
                            make_scorer)
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from lightgbm import LGBMClassifier
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight



In [10]:
#primary metric that will be used for model comparison
PRIMARY_METRIC = "Average Precision Score"

In [40]:
#loading config file
config_path = '../config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
    

# MLflow

In [12]:
experiment_name = config['pipeline']['experiment_name']

In [13]:
mlflow.set_experiment(experiment_name)


<Experiment: artifact_location='file:///home/rodolfo/Insync/rodolfopcruz2%40gmail.com/Google%20Drive/Estudo/Projetos-Novos/Credit_card_fraud_detection/notebooks/mlruns/240952246178253651', creation_time=1768924121470, experiment_id='240952246178253651', last_update_time=1768924121470, lifecycle_stage='active', name='Credit-Card-Fraud-Detection', tags={}>

In [69]:
mlflow.start_run(run_name='Model Selection')

Exception: Run with UUID b751b65ad92d4afa8933de080438d3e6 is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

In [41]:
artifacts_dir = Path(config['model_selection']['artifacts_path'])

# Load Dataset


In [20]:
dataset_path = Path(config['feature_engineering']['train_path_feature_engineered'])
target_column = config['data']['target_column']

In [21]:
df_train = pd.read_parquet(dataset_path)

In [22]:
y_train = df_train[target_column]

In [23]:
x_train = df_train.drop(columns = ['Time', target_column])

# Models

In [25]:
random_state = config['model_training']['random_state']

In [27]:
#models that will be tested
dummy = DummyClassifier(strategy = 'most_frequent')
lr = LogisticRegression(max_iter=1000,
                       random_state=random_state, verbose=False)
catboost = CatBoostClassifier(iterations=100, random_state=random_state, verbose=False)
xgboost = xgb.XGBClassifier(n_estimators=100, random_state=random_state)
adaboost = AdaBoostClassifier(n_estimators=100, random_state=random_state)
rf = RandomForestClassifier(n_estimators=100, random_state=random_state, verbose=False)
extra_tree = ExtraTreesClassifier(n_estimators=100 , random_state=random_state, verbose=False)
lgb = LGBMClassifier(n_estimators=100, random_state=random_state)
mlp = MLPClassifier(random_state=random_state, verbose=False)


In [28]:
candidate_models = {'Dummy': dummy,
                    'Logistic Regression': lr,
                   'CatBoost': catboost,
                   'XgBoost': xgboost,
                   'AdaBoost': adaboost,
                   'Random Forest': rf,
                   'Extra Trees': extra_tree,
                   'LightGBM': lgb, 
                   'MLP': mlp}

In [29]:
#metrics that will be used for model comparison
metrics = ['Balanced Accuracy Score',
           'Precision Score',
           'Recall Score',
           'F1 Score',
           'Average Precision Score',
           'Roc AUC']


In [30]:
scoring = {
    'balanced_accuracy': 'balanced_accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'average_precision': 'average_precision',
    'roc_auc': 'roc_auc'
}

In [31]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

# First Tests 

In [32]:
results_first_tests = pd.DataFrame(columns = metrics,
                       index= candidate_models.keys(),
                       data = 0.0)
results_first_tests

,Balanced Accuracy Score,Precision Score,Recall Score,F1 Score,Average Precision Score,Roc AUC
Dummy,0.0,0.0,0.0,0.0,0.0,0.0
Logistic Regression,0.0,0.0,0.0,0.0,0.0,0.0
CatBoost,0.0,0.0,0.0,0.0,0.0,0.0
XgBoost,0.0,0.0,0.0,0.0,0.0,0.0
AdaBoost,0.0,0.0,0.0,0.0,0.0,0.0
Random Forest,0.0,0.0,0.0,0.0,0.0,0.0
Extra Trees,0.0,0.0,0.0,0.0,0.0,0.0
LightGBM,0.0,0.0,0.0,0.0,0.0,0.0
MLP,0.0,0.0,0.0,0.0,0.0,0.0


## All selected models will be trained with their default parameter settings, except for the number of estimators used in the ensemble models.

In [33]:
for model_name, model in candidate_models.items():
    print(f'Training model {model_name}')

    cv_results = cross_validate(model, 
                                x_train, 
                                y_train, 
                                cv=cv, 
                                scoring=scoring, 
                                verbose = False)
   
    metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
    print(metrics_results)
    results_first_tests.loc[model_name, :] = metrics_results


Training model Dummy


/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rodolfo/.local/lib/python3.10/site-pa

[0.5, 0.0, 0.0, 0.0, 0.001666599482297844, 0.5]
Training model Logistic Regression
[0.8141461598363398, 0.8746606517002936, 0.6284486657620987, 0.7294916328458524, 0.7549685502501543, 0.9756132215557931]
Training model CatBoost
[0.9048669344709375, 0.9454779024062889, 0.8098145635459068, 0.8715345405653551, 0.8481817561400966, 0.9745286930051913]
Training model XgBoost
[0.894275760735497, 0.8863452300573329, 0.7887381275440977, 0.8307130797106446, 0.7481980645738604, 0.951166267948125]
Training model AdaBoost


/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
 

[0.8806645326694028, 0.877414695959778, 0.7615106286748077, 0.8141197253442325, 0.8120760949002953, 0.9652050233137743]
Training model Random Forest
[0.8791093587133615, 0.9409560651246135, 0.7582994120307553, 0.8393826315766946, 0.8386715950966902, 0.9455570078351642]
Training model Extra Trees
[0.885185013661812, 0.9306832642880309, 0.7704658525554049, 0.8429015914064035, 0.8403581215443717, 0.9499704026967916]
Training model LightGBM
[LightGBM] [Info] Number of positive: 265, number of negative: 158621
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005585 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7395
[LightGBM] [Info] Number of data points in the train set: 158886, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001668 -> initscore=-6.394543
[LightGBM] [Info] Start training from score -6.394543
[LightGBM] [Info] Number of positive: 265, number of negative: 15

In [35]:
results_first_tests.sort_values(by='Average Precision Score', ascending=False)

,Balanced Accuracy Score,Precision Score,Recall Score,F1 Score,Average Precision Score,Roc AUC
CatBoost,0.904867,0.945478,0.809815,0.871535,0.848182,0.974529
Extra Trees,0.885185,0.930683,0.770466,0.842902,0.840358,0.949970
Random Forest,0.879109,0.940956,0.758299,0.839383,0.838672,0.945557
AdaBoost,0.880665,0.877415,0.761511,0.814120,0.812076,0.965205
MLP,0.889668,0.883634,0.779512,0.827521,0.810285,0.945964
Logistic Regression,0.814146,0.874661,0.628449,0.729492,0.754969,0.975613
XgBoost,0.894276,0.886345,0.788738,0.830713,0.748198,0.951166
LightGBM,0.744728,0.218236,0.492311,0.299320,0.173629,0.653817
Dummy,0.500000,0.000000,0.000000,0.000000,0.001667,0.500000


In [43]:
results_first_tests.to_parquet(artifacts_dir / 'first_tests.parquet')

In [44]:
mlflow.log_artifact(
        artifacts_dir / "first_tests.parquet",
        artifact_path="model_selection"
    )

# Second Tests

## Adjusting a hyperparameter in certain models to address the class imbalance in the dataset.

In [45]:
candidate_models_second_tests = [
                   'Dummy',
                   'Logistic Regression',
                   'CatBoost',
                   'XgBoost',
                   'AdaBoost',
                   'Random Forest',
                   'Extra Trees',
                   'LightGBM']

In [46]:
results_second_tests = pd.DataFrame(columns = metrics,
                       index= candidate_models_second_tests,
                       data = 0.0)
results_second_tests

,Balanced Accuracy Score,Precision Score,Recall Score,F1 Score,Average Precision Score,Roc AUC
Dummy,0.0,0.0,0.0,0.0,0.0,0.0
Logistic Regression,0.0,0.0,0.0,0.0,0.0,0.0
CatBoost,0.0,0.0,0.0,0.0,0.0,0.0
XgBoost,0.0,0.0,0.0,0.0,0.0,0.0
AdaBoost,0.0,0.0,0.0,0.0,0.0,0.0
Random Forest,0.0,0.0,0.0,0.0,0.0,0.0
Extra Trees,0.0,0.0,0.0,0.0,0.0,0.0
LightGBM,0.0,0.0,0.0,0.0,0.0,0.0


In [47]:
for model_name in candidate_models_second_tests:

    if model_name == 'Dummy':
        model_adjusted = DummyClassifier(strategy = 'most_frequent')
    
    elif model_name == 'Logistic Regression':
        model_adjusted = LogisticRegression(max_iter=5000,
                                           random_state=random_state,
                                           class_weight='balanced', verbose=False)        
    
    elif model_name == 'CatBoost':
        model_adjusted = CatBoostClassifier(iterations=100,
                                            auto_class_weights='Balanced',
                                            random_state=random_state, verbose=False)

    elif model_name == 'XgBoost':
        n_major = sum(y_train==0)
        n_minor = sum(y_train==1)
        scale_pos_weight = n_major / n_minor

        model_adjusted = xgb.XGBClassifier(n_estimators=100, 
                                           random_state=random_state,
                                           scale_pos_weight=scale_pos_weight)
    elif model_name == 'AdaBoost':
        base_tree = DecisionTreeClassifier(
                    max_depth=1,
                    class_weight='balanced',
                    random_state=random_state)
        model_adjusted = AdaBoostClassifier(n_estimators=100, 
                                      estimator=base_tree,
                                      random_state=random_state)

    elif model_name == 'Random Forest':
        model_adjusted = RandomForestClassifier(n_estimators=100, 
                                    class_weight='balanced',
                                    random_state=random_state,
                                    verbose=False)
                                            
    elif model_name == 'Extra Trees':
        model_adjusted = ExtraTreesClassifier(n_estimators=100,
                                          class_weight='balanced',
                                          random_state=random_state,
                                          verbose=False)                           
    elif model_name == 'LightGBM':
        model_adjusted = LGBMClassifier(n_estimators=100, 
                        random_state=random_state,
                        class_weight='balanced')

   
                                              
    
    print(f'Training model {model_name}')
    cv_results = cross_validate(model_adjusted, x_train, y_train, cv=cv, scoring=scoring, verbose=False)
    metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
    print(metrics_results)
    results_second_tests.loc[model_name, :] = metrics_results
    

Training model Dummy


/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rodolfo/.local/lib/python3.10/site-pa

[0.5, 0.0, 0.0, 0.0, 0.001666599482297844, 0.5]
Training model Logistic Regression
[0.9401258349388737, 0.05480252951376911, 0.9065128900949796, 0.10331120931317914, 0.7525172713301751, 0.9815764919657527]
Training model CatBoost
[0.9182689398457651, 0.7552692857984481, 0.8369968340117595, 0.7932620964307155, 0.8361983606724269, 0.9633241986202968]
Training model XgBoost
[0.9108243939337883, 0.9032190760059613, 0.8218000904568068, 0.8598454118854978, 0.8548965507217696, 0.9811360102685436]
Training model AdaBoost


/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/home/rodolfo/.local/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
 

[0.9308082751854491, 0.20988755326458325, 0.8671189507010402, 0.33744771651105765, 0.7943041284956001, 0.9662979557590999]
Training model Random Forest
[0.8701189931753429, 0.953359477124183, 0.7402985074626866, 0.8332478632478633, 0.8369642690930871, 0.9457921624199784]
Training model Extra Trees
[0.885242850688541, 0.9478312537136067, 0.7705563093622795, 0.8498193024233085, 0.8413894498481497, 0.9456936366591823]
Training model LightGBM
[LightGBM] [Info] Number of positive: 265, number of negative: 158621
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005567 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7395
[LightGBM] [Info] Number of data points in the train set: 158886, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Info] Number of positive: 265, number of negative: 15

In [49]:
results_second_tests.sort_values(by=PRIMARY_METRIC, ascending=False) 

,Balanced Accuracy Score,Precision Score,Recall Score,F1 Score,Average Precision Score,Roc AUC
XgBoost,0.910824,0.903219,0.821800,0.859845,0.854897,0.981136
Extra Trees,0.885243,0.947831,0.770556,0.849819,0.841389,0.945694
Random Forest,0.870119,0.953359,0.740299,0.833248,0.836964,0.945792
LightGBM,0.912337,0.874217,0.824876,0.848251,0.836757,0.966188
CatBoost,0.918269,0.755269,0.836997,0.793262,0.836198,0.963324
AdaBoost,0.930808,0.209888,0.867119,0.337448,0.794304,0.966298
Logistic Regression,0.940126,0.054803,0.906513,0.103311,0.752517,0.981576
Dummy,0.500000,0.000000,0.000000,0.000000,0.001667,0.500000


In [50]:
results_second_tests.to_parquet(artifacts_dir / 'second_tests.parquet')

In [51]:
mlflow.log_artifact(
        artifacts_dir / "second_tests.parquet",
        artifact_path="model_selection"
    )

# Comparison

## Combining the results of both tests into a single DataFrame

In [53]:
comparison = pd.concat(
    [results_first_tests, results_second_tests],
    axis=1,
    join='inner',
    keys=['First Tests', 'Second Tests']
)

In [54]:
comparison.columns.names = ['Tests', 'Metrics']


In [59]:
comparison = comparison.sort_values(by=[('First Tests','Average Precision Score')], ascending=False)
comparison

In [61]:
file_name = 'comparison.parquet'
comparison.to_parquet(artifacts_dir / file_name)
mlflow.log_artifact(
        artifacts_dir / file_name,
        artifact_path="model_selection"
    )

## Percentage difference between the second and first tests

In [62]:
diff_pct = ((comparison['Second Tests'] - comparison['First Tests'])/ comparison['First Tests'])*100

In [63]:
diff_pct.columns = [f'{column_name} (%)' for column_name in diff_pct.columns]

In [64]:
diff_pct.style.format('{:.2f}')

,Balanced Accuracy Score (%),Precision Score (%),Recall Score (%),F1 Score (%),Average Precision Score (%),Roc AUC (%)
CatBoost,1.48,-20.12,3.36,-8.98,-1.41,-1.15
Extra Trees,0.01,1.84,0.01,0.82,0.12,-0.45
Random Forest,-1.02,1.32,-2.37,-0.73,-0.20,0.02
AdaBoost,5.69,-76.08,13.87,-58.55,-2.19,0.11
Logistic Regression,15.47,-93.73,44.25,-85.84,-0.32,0.61
XgBoost,1.85,1.90,4.19,3.51,14.26,3.15
LightGBM,22.51,300.58,67.55,183.39,381.92,47.78
Dummy,0.00,nan,nan,nan,0.00,0.00


In [68]:
file_name = 'percentage_difference.parquet'
diff_pct.to_parquet(artifacts_dir / file_name)
mlflow.log_artifact(
        artifacts_dir / file_name,
        artifact_path="model_selection"
    )

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [66]:
mlflow.end_run()

[2026-01-20 15:20:54 -0300] [97832] [INFO] Starting gunicorn 23.0.0
[2026-01-20 15:20:54 -0300] [97832] [INFO] Listening at: http://127.0.0.1:5000 (97832)
[2026-01-20 15:20:54 -0300] [97832] [INFO] Using worker: sync
[2026-01-20 15:20:54 -0300] [97833] [INFO] Booting worker with pid: 97833
[2026-01-20 15:20:54 -0300] [97834] [INFO] Booting worker with pid: 97834
[2026-01-20 15:20:55 -0300] [97836] [INFO] Booting worker with pid: 97836
[2026-01-20 15:20:55 -0300] [97837] [INFO] Booting worker with pid: 97837
^C
[2026-01-20 15:36:00 -0300] [97832] [INFO] Handling signal: int
[2026-01-20 15:36:00 -0300] [97834] [INFO] Worker exiting (pid: 97834)
[2026-01-20 15:36:00 -0300] [97837] [INFO] Worker exiting (pid: 97837)
[2026-01-20 15:36:00 -0300] [97833] [INFO] Worker exiting (pid: 97833)
[2026-01-20 15:36:00 -0300] [97836] [INFO] Worker exiting (pid: 97836)
